In [97]:
import ollama
import json
import re

In [98]:
with open("data/grids.json", "r") as f:
    grids = json.load(f)

In [99]:
MODEL = "mistral"

In [100]:
SYSTEM_PROMPT = """
You are a robot path planner operating in a 5×5 grid world.

Environment
    - Coordinates are represented as [row, column].
    - Rows and columns are indexed from 0 to 4.
    - Row indices increase from top to bottom.
    - Column indices increase from left to right.

Movement rules:
    - NORTH: [row - 1, column]
    - SOUTH: [row + 1, column]
    - EAST:  [row, column + 1]
    - WEST:  [row, column - 1]
    
    Each action moves the robot exactly one cell. Only the specified coordinate changes for each action.

Constraints
    - Never enter an obstacle cell.
    - Never leave the grid boundaries.
    - Each action moves exactly one cell.
    - Return the possible next action.
    - Any valid action is acceptable.

Planning procedure:
    1. Start at the given starting position.
    2. Select an action that moves toward the goal while avoiding obstacles.
    3. Never choose an action that leaves the grid or enters an obstacle.
    4. A feedback will be provided after each action to indicate whether the action was valid or invalid.
    5. If the action was valid, plan the next action based on the robot position.
    6. If the action was invalid, plan a different action.

Output format
    - Return only a valid JSON object.
    - Example response: {"action": "SOUTH"}
    - Do not include explanations, Markdown fences, intermediate positions, or additional text.

"""

In [101]:
def ask_slm(messages):
    # print("Asking SLM...")
    response = ollama.chat(
        model=MODEL,
        messages=messages
    )

    return response["message"]["content"]

In [102]:
def validate_json(json_str):
    json_str = json_str.strip()
    json_str = re.sub(r"^```(?:json)?\s*", "", json_str)
    json_str = re.sub(r"\s*```$", "", json_str)
    try:
        data = json.loads(json_str)
        if isinstance(data, dict) and "action" in data:
            return True, data['action']
        else:
            return False, "INVALID_JSON"
    except json.JSONDecodeError:
        return False, "INVALID_JSON"

In [103]:
MOVES = {
    "NORTH": (-1, 0),
    "SOUTH": (1, 0),
    "EAST": (0, 1),
    "WEST": (0, -1),
}


def validate_move(position, action, obstacles, grid_size=5):
    action = action.upper()

    if action not in MOVES:
        return {
            "valid": False,
            "reason": "Unknown action"
        }

    row, col = position
    dr, dc = MOVES[action]
    next_position = (row + dr, col + dc)

    if not (
        0 <= next_position[0] < grid_size
        and 0 <= next_position[1] < grid_size
    ):
        return {
            "valid": False,
            "reason": "Move leaves grid boundaries"
        }

    if next_position in {tuple(p) for p in obstacles}:
        return {
            "valid": False,
            "reason": "Move enters an obstacle"
        }

    return {
        "valid": True,
        "next_position": list(next_position)
    }

In [104]:
solved_grids = 0

for grid_info in grids[:1]:
    grid = grid_info["grid"]

    start_position = grid_info["start"]
    current_position = start_position
    goal_position = grid_info["goal"]
    obstacles = grid_info["obstacles"]
    total_moves = 0
    attempted_moves = 0
    is_solved = False

    

    print(f"Grid:\n{'\n'.join(' '.join(row) for row in grid)}")

    initial_prompt = f"Starting position: {start_position}\nGoal position: {goal_position}\nObstacles: {obstacles}"
    


    messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": initial_prompt}
        ]


    while attempted_moves < 25:
        attempted_moves += 1
        system_message = messages[0]
        initial_user_message = messages[1]
        recent_messages = messages[2:][-4:]

        context = [
            system_message,
            initial_user_message,
            *recent_messages
        ]

        response = ask_slm(context)
        
        is_valid, action = validate_json(response)

        if not is_valid:
            print(f"Invalid JSON response: {response}. Retrying...")


            messages.append({
                "role": "user",
                "content": (
                    f"The action was invalid.\n"
                    f"Current position: {current_position}\n"
                    "Please provide a valid JSON object with an action."
                ),
            })

            continue

        validation_result = validate_move(current_position, action, obstacles)

        if validation_result["valid"]:
            total_moves += 1
            # print(f"Valid move: {action}.")
            current_position = validation_result['next_position']
            is_solved = current_position == goal_position

            if is_solved:
                solved_grids += 1
                break

            messages.append({
                "role": "user",
                "content": (
                    f"The action was valid.\n"
                    f"Current position: {current_position}\n"
                    f"Goal position: {goal_position}\n"
                    f"Obstacles: {obstacles}\n"
                ),
            })

            
        else:
            messages.append({
                "role": "user",
                "content": f"Invalid move: {action}.\nCurrent position: {current_position}.\n Reason: {validation_result['reason']}. Retrying..."
            })

    # print(f"Response:\n{response}\n")
    print(f"Total attempted moves: {attempted_moves}")
    print(f"Total executed moves: {total_moves}")

    if is_solved:
        print("Goal reached!")
    else:
        print("Failed to reach the goal within 25 attempts.")

print(f"Total solved grids: {solved_grids} out of {len(grids[:1])}")

Grid:
S . . . B
B . . . .
. G . . .
. . . . B
. . . B .
Total attempted moves: 8
Total executed moves: 7
Goal reached!
Total solved grids: 1 out of 1
